<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec02_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec02_vision.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第2回 視覚特性と画像入出力
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，scikit-image，Matplotlib，colour-demosaicing

MATLAB のライブスクリプト `code/matlab/vie_sec02_vision.m` を Python に移植したもの。第2回のスライド（vie2026-02）で使う図と数値を作る。視覚特性のデモ画像，網膜像の大きさ，視力と走査線数，量子化，RGB 成分，フレームレートとビットレート，アレイセンサによる標本化，ベイヤ配列を順に扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

記号は教科書（村松正吾『多次元信号・画像処理の基礎と展開』）に合わせる。図の配色は MSIP Lab のロゴの 3 色（緑 #008855，青 #2E75B6，橙 #C55A11）と灰色を使う（色そのものが内容の画像は除く）。

画像処理には scikit-image，NumPy，colour-demosaicing を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。写真は教科書のサンプル画像 msipimg01〜08（512×512 のカラー）に限り，`vie.msipimg` で読み込む（`data` フォルダに無ければ取得する）。

In [ ]:
import os
import urllib.request

if not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")
try:
    import colour_demosaicing
except ImportError:
    %pip install -q colour-demosaicing
    import colour_demosaicing

import numpy as np
import matplotlib.pyplot as plt
from skimage.util import img_as_float, img_as_ubyte
from skimage.metrics import peak_signal_noise_ratio
import vie

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

図に使うロゴの 3 色と，それを白と混ぜた淡い色を用意する。

In [ ]:
ccool_light = 0.35 * np.array(ccool) + 0.65   # 寒色系の塗り

## 視覚特性のデモ：明度対比
背景の明るさだけを変え，中央の正方形は**同じ画素値**にする。同じ明るさなのに，背景が暗いほど明るく見える。

In [ ]:
bg = [0, 0.4, 0.8]          # 背景の明るさ（黒・灰・明るい灰）
fg = 0.55                   # 中央の正方形の明るさ（3 枚とも同じ）
sz, w = 120, 50
panels = []
for b in bg:
    P = b * np.ones((sz, sz))
    P[(sz - w) // 2:(sz + w) // 2, (sz - w) // 2:(sz + w) // 2] = fg
    panels.append(P)
sep = np.ones((sz, 20))
C = np.hstack([panels[0], sep, panels[1], sep, panels[2]])

fig, ax = plt.subplots(figsize=(7, 2.2), layout="constrained")
show(ax, C)
plt.show()

中央の画素値を確かめると，3 枚ともまったく同じである。

In [ ]:
centers = np.array([P[sz // 2, sz // 2] for P in panels])
print("centers =", centers)
print(f"fg = {fg:.2f}")

スライドでは，まず背景をすべて白にした版を見せ，中央の正方形が同じ明るさに見えることを確かめてから，背景を切り替える（明暗の順応と対比の体験）。

In [ ]:
Cw = C.copy()
for k in range(len(bg)):
    cols = slice(k * (sz + 20), k * (sz + 20) + sz)
    blk = Cw[:, cols]
    blk[blk != fg] = 1                       # 背景だけを白にする
    Cw[:, cols] = blk

fig, ax = plt.subplots(figsize=(7, 2.2), layout="constrained")
show(ax, Cw)
ax.set_axis_on()                             # 白い背景が分かるよう枠を付ける
ax.set(xticks=[], yticks=[])
plt.show()

## 視覚特性のデモ：色順応
左に青みがかった画像，右に黄みがかった画像を並べ，間に白い点を置く。白い点をしばらく見つめると，目が色に順応する。

画像は msipimg02（花束）を 384×384 に縮小して使う。白いカーネーションが含まれるので，順応の前後で白の見え方が変わるのが分かりやすい。

In [ ]:
X = img_as_float(vie.msipimg(2, 384))
tintB = X * np.array([0.55, 0.65, 1.00])         # 青みがかった画像
tintY = X * np.array([1.00, 0.90, 0.45])         # 黄みがかった画像
gap = np.zeros((X.shape[0], 24, 3))
D = np.hstack([tintB, gap, tintY])
r = (D.shape[0] - 1) // 2                        # 白い点の位置（中央）
c = X.shape[1] + (gap.shape[1] - 1) // 2
rr, cc = np.mgrid[0:D.shape[0], 0:D.shape[1]]
dot = (cc - c) ** 2 + (rr - r) ** 2 <= 16
D[dot] = 1

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
show(ax, np.clip(D, 0, 1))
plt.show()

順応させるための画面：左を青一色，右を黄色一色で覆い，同じ位置に白い点を置く。スライドではこれをしばらく見つめたあと，上の画像に切り替える。

In [ ]:
blue = np.array([0.10, 0.25, 1.00])
yellow = np.array([1.00, 0.88, 0.10])
A = np.hstack([np.broadcast_to(blue, X.shape), gap, np.broadcast_to(yellow, X.shape)])
A[dot] = 1

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
show(ax, A)
plt.show()

## 可視光のスペクトル
波長 380 nm〜780 nm の単色光の色を，近似式（D. Bruton による波長→RGB の変換）で描く。短波長側から紫・青・緑・黄・橙・赤と移り変わり，両端では視感度が下がるので暗くなる。

In [ ]:
def wl2rgb(lam):
    """波長 [nm] を近似的な表示色（RGB，0〜1）に変換する（D. Bruton の近似式）．"""
    if lam < 440:
        c = [-(lam - 440) / (440 - 380), 0, 1]
    elif lam < 490:
        c = [0, (lam - 440) / (490 - 440), 1]
    elif lam < 510:
        c = [0, 1, -(lam - 510) / (510 - 490)]
    elif lam < 580:
        c = [(lam - 510) / (580 - 510), 1, 0]
    elif lam < 645:
        c = [1, -(lam - 645) / (645 - 580), 0]
    else:
        c = [1, 0, 0]
    # 視感度の低い両端を暗くする
    if lam < 420:
        f = 0.3 + 0.7 * (lam - 380) / (420 - 380)
    elif lam > 700:
        f = 0.3 + 0.7 * (780 - lam) / (780 - 700)
    else:
        f = 1
    return (f * np.array(c, dtype=float)) ** 0.8


lam = np.linspace(380, 780, 801)
rgb = np.array([wl2rgb(l) for l in lam])
Sp = np.broadcast_to(rgb[None, :, :], (60, lam.size, 3))

fig, ax = plt.subplots(figsize=(8, 1.4), layout="constrained")
ax.imshow(Sp, extent=(lam[0], lam[-1], 0, 1), aspect="auto")
ax.set(yticks=[], xlabel="波長 [nm]")
plt.show()

## 網膜像の大きさ
100 m 先にある高さ 15 m の木を見るとき，水晶体の中心から網膜までを約 17 mm とすると，網膜像の高さ $h$ は相似な三角形から求まる。

$$\frac{15}{100} = \frac{h}{17}$$

In [ ]:
treeH, treeDist, lensToRetina = 15, 100, 17   # [m], [m], [mm]
h = lensToRetina * treeH / treeDist            # 網膜像の高さ [mm]
print(f"h = {h:.2f} mm")

## 視力と解像度
**視力**は，隣接する 2 点を弁別できる最小視角の逆数である（視力 = 1/視角〔分〕）。視力 $V$ の人は $1/V$ 分（1 分 = 1/60 度）の視角を見分けられる。

画面の高さを $h$ とし，画面高の 3 倍の距離 $d=3h$ からテレビを見る（2025 年度の演習課題（2）-2 と同じ設定）。画面の上端と下端が目につくる視角 $\theta$ は，直角三角形 2 つに分けて

$$\theta = 2\tan^{-1}\frac{h/2}{d} = 2\tan^{-1}\frac{h/2}{3h} = 2\tan^{-1}\frac{1}{6}$$

In [ ]:
dh = 3                                     # 視距離と画面高の比 d/h
theta = 2 * np.degrees(np.arctan(0.5 / dh))   # 視角 [度]
arcmin = theta * 60                        # 視角 [分]
print(f"θ = {theta:.4f} 度 = {arcmin:.4f} 分")

視力 $V$ の人は $1/V$ 分ずつ見分けられるので，画面の縦方向に必要な走査線の数は，視角 $\theta$〔分〕を $1/V$〔分〕で割った値が目安になる。

In [ ]:
V = 1                                      # 視力
nlines = arcmin / (1 / V)                  # 走査線数の目安 [本]
print(f"nlines = {nlines:.4f} 本")

視力 1 なら約 1135 本。ハイビジョン（有効走査線数 1080 本）の走査線数の目安になる。

In [ ]:
print(f"θ ≃ {theta:.1f} 度，{arcmin:.0f} 分，走査線数 {nlines:.0f} 本")

位置関係を縮尺どおりに描く（2025 年度の演習課題（2）-2 の図を参考）。画面の高さを $h=1$ とし，目の節点（光線が交わる点）を画面から $d=3h$ の位置に置く。

In [ ]:
def dimarrow(ax, p1, p2, col, hl=0.07, hw=0.03):
    """両端に矢じりのある寸法線を p1 から p2 へ描く（データ座標）．

    hl, hw は矢じりの長さと半幅（データ単位）．
    """
    p1, p2 = np.asarray(p1, float), np.asarray(p2, float)
    u = (p2 - p1) / np.linalg.norm(p2 - p1)   # 向き
    nv = np.array([-u[1], u[0]])               # 法線
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=col, lw=1.2)
    for tip, d in ((p1, -u), (p2, u)):
        base = tip - hl * d
        v = np.array([tip, base + hw * nv, base - hw * nv])
        ax.add_patch(plt.Polygon(v, closed=True, fc=col, ec=col))


fig, ax = plt.subplots(figsize=(6, 2.3), layout="constrained")
th = np.radians(theta)                         # 視角 [rad]
node = (dh, 0)                                 # 目の節点
# 画面（高さ h）
ax.add_patch(plt.Polygon([[-0.07, -0.5], [0, -0.5], [0, 0.5], [-0.07, 0.5]],
                         fc=ccool_light, ec=ccool, lw=1.5))
# 眼球と水晶体
tt = np.linspace(0, 2 * np.pi, 200)
ax.plot(dh + 0.30 + 0.42 * np.cos(tt), 0.42 * np.sin(tt), color=cmain, lw=3)
ax.plot(dh - 0.10 + 0.055 * np.cos(tt), 0.17 * np.sin(tt), color=cmain, lw=2)
# 画面の上端・下端から節点を通って網膜へ向かう光線
xe = dh + 0.62
ax.plot([0, xe], [0.5, 0.5 - xe / (2 * dh)], color=(0.3, 0.3, 0.3), lw=1)
ax.plot([0, xe], [-0.5, -0.5 + xe / (2 * dh)], color=(0.3, 0.3, 0.3), lw=1)
ax.plot(*node, "o", ms=6, color=cmain)
# 視角 θ の円弧
ra = 1.0
ta = np.linspace(np.pi - th / 2, np.pi + th / 2, 50)
ax.plot(node[0] + ra * np.cos(ta), node[1] + ra * np.sin(ta), color=cwarm, lw=2.5)
ax.text(node[0] - ra - 0.08, 0, r"$\theta$", fontsize=22, color=cwarm, ha="right", va="center")
# 寸法線 h と d = 3h
ax.plot([-0.45, -0.1], [0.5, 0.5], color=(0.5, 0.5, 0.5))
ax.plot([-0.45, -0.1], [-0.5, -0.5], color=(0.5, 0.5, 0.5))
dimarrow(ax, (-0.32, -0.5), (-0.32, 0.5), "k")
ax.text(-0.40, 0, "$h$", fontsize=22, ha="right", va="center")
ax.plot([0, 0], [-0.55, -0.85], color=(0.5, 0.5, 0.5))
ax.plot([dh, dh], [-0.1, -0.85], color=(0.5, 0.5, 0.5))
dimarrow(ax, (0, -0.75), (dh, -0.75), "k")
ax.text(dh / 2, -0.75, "$d=3h$", fontsize=22, ha="center", va="center",
        bbox=dict(fc="w", ec="none", pad=1))
ax.set_aspect("equal")
ax.set_axis_off()
ax.set(xlim=(-0.75, 3.85), ylim=(-0.95, 0.6))
plt.show()

## 階調と量子化
量子化は振幅の離散化である。教科書 2.1.6 項の**線形量子化**は，量子化ステップ $Q$ を用いて

$$y=\phi(x)=\left\lfloor \frac{x}{Q} \right\rceil,\quad \check{x}=\breve{\phi}(y)=Qy$$

と表される（$\lfloor\cdot\rceil$ は四捨五入，$\check{x}$ は逆量子化の結果）。画素値 $x\in[0,1]$ を $L=2^\beta$ 階調で表すときは $Q=(2^\beta-1)^{-1}$ とする。$\beta$ を小さくすると，緩やかな濃淡に**擬似輪郭**が現れる。

画像は msipimg05（石像の顔）のグレースケールを 256×256 に縮小して使う。空のなめらかな階調に擬似輪郭が現れ，石の細部との違いが分かりやすい。

In [ ]:
Xg = img_as_float(vie.msipimg(5, 256, "gray"))
betas = np.array([1, 2, 8])                      # 1 画素あたりのビット数 β
emax = np.zeros(betas.size)
fig, axs = plt.subplots(2, betas.size, figsize=(9, 6.4), layout="constrained")
for k, bt in enumerate(betas):
    L = 2 ** bt                                  # 階調数
    Q = 1 / (L - 1)                              # 量子化ステップ
    y = np.floor(Xg / Q + 0.5)                     # 線形量子化（整数 0,1,…,L-1）
    Xq = Q * y                                   # 逆量子化
    e = Xg - Xq                                  # 量子化誤差
    emax[k] = np.abs(e).max()
    show(axs[0, k], Xq, f"{L} 階調（{bt} bit）")
    show(axs[1, k], np.abs(e) / (Q / 2), "|誤差|（Q/2 で正規化）")
plt.show()
levels = 2 ** betas
print("levels =", levels)

四捨五入なので，量子化誤差の大きさは $Q/2$ を超えない。

In [ ]:
print(f"{'β':>3} {'階調数':>6} {'Q':>10} {'max|e|':>10}")
for bt, L, em in zip(betas, levels, emax):
    print(f"{bt:>3} {L:>8} {1 / (L - 1):>10.4f} {em:>10.4g}")

量子化特性（入力 $x$ と逆量子化の結果 $\check{x}$ の関係）を $\beta=2$（4 階調）について描くと階段状になる。破線は $\check{x}=x$。

In [ ]:
beta2 = 2
Q2 = 1 / (2 ** beta2 - 1)
xx = np.linspace(0, 1, 3001)
fig, ax = plt.subplots(figsize=(3.6, 3.4), layout="constrained")
ax.plot(xx, xx, "--", color=(0.55, 0.55, 0.55), lw=1.2)
ax.plot(xx, Q2 * np.floor(xx / Q2 + 0.5), color=cmain, lw=3)
# 量子化ステップ Q を示す矢印（破線と交わらない位置に置く）
dimarrow(ax, (0.76, Q2), (0.76, 2 * Q2), cwarm, 0.045, 0.018)
ax.text(0.73, 1.5 * Q2, "$Q$", fontsize=16, color=cwarm, ha="right", va="center")
tk = np.arange(2 ** beta2) * Q2
ax.set(xlim=(0, 1), ylim=(0, 1), xticks=tk, yticks=tk,
       xticklabels=["$0$", "$Q$", "$2Q$", "$3Q$"], yticklabels=["$0$", "$Q$", "$2Q$", "$3Q$"])
ax.set_aspect("equal")
ax.grid(True)
ax.set_xlabel("$x$", fontsize=14)
# Matplotlib の数式では \check の位置がずれるので，\overset{\vee}{x} で代用する
ax.set_ylabel(r"$\overset{\vee}{x}=Q\lfloor x/Q\rceil$", fontsize=14)
ax.tick_params(labelsize=12)
plt.show()

## 色の表し方：RGB 表色系
画素 $\boldsymbol{n}$ の色 $\vec{x}[\boldsymbol{n}]$ を 3 原色の単位ベクトル $\vec{e}_\mathrm{R},\vec{e}_\mathrm{G},\vec{e}_\mathrm{B}$ のベクトル和で表す（記号は教科書 2.2.1 項の $\vec{x}[\boldsymbol{n}]=(x_\mathrm{R}[\boldsymbol{n}]\ x_\mathrm{G}[\boldsymbol{n}]\ x_\mathrm{B}[\boldsymbol{n}])^\top$ に合わせる。この回ではビット数 $B$ とビットレート $R$ も使うので，成分を $R,G,B$ とは書かない）。

$$\vec{x}[\boldsymbol{n}] = x_\mathrm{R}[\boldsymbol{n}]\,\vec{e}_\mathrm{R} + x_\mathrm{G}[\boldsymbol{n}]\,\vec{e}_\mathrm{G} + x_\mathrm{B}[\boldsymbol{n}]\,\vec{e}_\mathrm{B}$$

画像は msipimg02（花束）を 384×384 に縮小し，下側の 256×384 画素（赤・橙・黄・白の花が並ぶ部分）を切り出して使う。花の色ごとに R, G, B の成分の強さが異なる。

In [ ]:
Y = img_as_float(vie.msipimg(2, 384))
Y = Y[128:384]
Z = np.zeros(Y.shape[:2])
Yr = np.stack([Y[:, :, 0], Z, Z], axis=2)
Yg = np.stack([Z, Y[:, :, 1], Z], axis=2)
Yb = np.stack([Z, Z, Y[:, :, 2]], axis=2)
fig, axs = plt.subplots(1, 4, figsize=(12, 2.6), layout="constrained")
for ax, im, ttl in zip(axs, (Y, Yr, Yg, Yb), ("フルカラー", "R", "G", "B")):
    show(ax, im, ttl)
plt.show()

ひとつの画素を取り出すと，3 つの数の組 $(x_\mathrm{R}\ x_\mathrm{G}\ x_\mathrm{B})$ になっている。

In [ ]:
pr, pc = 201, 109                 # 左下の赤い花（ダリア）のあたり
rgb = Y[pr, pc, :]
print("rgb =", rgb)
print("({:.2f}, {:.2f}, {:.2f})".format(*rgb))

## フレームレート
フレームレート $\Delta_\mathrm{t}^{-1}$（fps）は 1 秒あたりのフレーム数。フレーム間隔 $\Delta_\mathrm{t}$ はその逆数である。

In [ ]:
fps = np.array([30, 24])           # テレビ，映画
interval = 1000 / fps              # フレーム間隔 [ms]
print("interval =", interval)
print(f"テレビ {interval[0]:.1f} ms，映画 {interval[1]:.1f} ms")

## 動画像のビットレート（教科書 1.1.2 項の例題）
画素数 $N_1\times N_2=4320\times 7680$，フレーム間隔 $\Delta_\mathrm{t}=1/60$ s，一配列要素あたり $\beta=8$ bits の RGB カラー動画像のビットレート $R$ を求める。1 フレームあたりのビット数は

$$B=3\beta N_1N_2$$

で，これにフレームレート $\Delta_\mathrm{t}^{-1}$ を掛けると $R=B\Delta_\mathrm{t}^{-1}$ [bps] になる。

In [ ]:
N1, N2 = 4320, 7680                # 画素数（8K）
nbits = 8                          # β
frameRate = 60                     # フレームレート 1/Δt [1/s]
bpp = 3 * nbits                    # 1 画素あたりのビット数 [bpp]
Npix = N1 * N2                     # 1 フレームあたりの画素数
Bframe = 3 * nbits * N1 * N2       # 1 フレームあたりのビット数 [bits/frame]
Rbps = Bframe * frameRate          # ビットレート [bps]
RGbps = Rbps / 1e9                 # [Gbps]
print(f"bpp = {bpp}，Npix = {Npix:,}，Bframe = {Bframe:,}，Rbps = {Rbps:,}，RGbps = {RGbps}")

教科書の解答（$B=796262400$ bits/frame，$R=47775744000$ bps $\simeq 48$ Gbps）と一致することを確かめる。

In [ ]:
assert Bframe == 796262400 and Rbps == 47775744000, "教科書の値と一致しない"
print(f"B = {Bframe:,} bits/frame，R = {Rbps:,} bps ≃ {RGbps:.0f} Gbps")

## 画像の取得：アレイセンサによる標本化
教科書 1.4.1 項の例「アレイセンサ」では，センサ上に結像した光の強度分布 $u(q_\mathrm{v},q_\mathrm{h})$ から，撮像装置の点光源応答 $\breve{\varphi}$ を介して，標本間隔 $\Delta_\mathrm{v},\Delta_\mathrm{h}$ ごとに配列の要素を得る。

$$x[n_\mathrm{v},n_\mathrm{h}]=\iint_{\mathbb{R}^2}\breve{\varphi}(\Delta_\mathrm{v}n_\mathrm{v}-q_\mathrm{v},\Delta_\mathrm{h}n_\mathrm{h}-q_\mathrm{h})\,u(q_\mathrm{v},q_\mathrm{h})\,\mathrm{d}q_\mathrm{v}\mathrm{d}q_\mathrm{h}$$

ここでは高解像度の画像を連続な強度分布とみなし，$\breve{\varphi}$ を一辺 $\Delta$ 画素の矩形（受光面の中で光を平均する）として，標本間隔 $\Delta=\Delta_\mathrm{v}=\Delta_\mathrm{h}$ を広げたときの観測をまねる。標本間隔が広いほど細部が失われる。

画像は細い線状の構造が多い msipimg07（モンブラン）のグレースケールを使う。

In [ ]:
Xs = img_as_float(vie.msipimg(7, None, "gray"))
deltas = [1, 4, 8]                               # 標本間隔 Δ（元の画素単位）
fig, axs = plt.subplots(1, len(deltas), figsize=(12, 4.3), layout="constrained")
for ax, dlt in zip(axs, deltas):
    nv, nh = Xs.shape[0] // dlt, Xs.shape[1] // dlt
    blocks = Xs[:nv * dlt, :nh * dlt].reshape(nv, dlt, nh, dlt)
    Xa = blocks.mean(axis=(1, 3))                # Δ×Δ の受光面で平均して標本化
    show(ax, np.kron(Xa, np.ones((dlt, dlt))), f"Δ = {dlt}（{nv}×{nh} 画素）")
plt.show()

## カラー撮像：単板式（原色ベイヤ形）
単板式では，感光部の前に色フィルタを市松状に並べる。ひとつの画素は R, G, B のどれか 1 色しか測れない。2×2 画素のうち G が 2 つ，R と B が 1 つずつである。

In [ ]:
Nsens = 4000 * 3000                # 例：約 1200 万画素のセンサ
counts = Nsens * np.array([1 / 4, 1 / 2, 1 / 4])   # R, G, B の画素数
print("counts =", counts)
print(f"{Nsens // 10**4} 万画素：R {counts[0] / 1e4:.0f} 万，G {counts[1] / 1e4:.0f} 万")

欠けた色は周囲から補間する（デモザイキング）。原画像からベイヤ配列の観測を作り，Malvar らの勾配補正線形補間（colour-demosaicing の `demosaicing_CFA_Bayer_Malvar2004`）で戻してみる。

画像は msipimg02（花束）の原寸から，黄色い花弁に細い暗い斑点が並ぶ 80×80 画素を切り出す。斑点の縁に補間による色のにじみ（ジッパー状の偽色）が現れる。

In [ ]:
Xb = vie.msipimg(2)
crop = Xb[180:260, 380:460, :]                     # 一部を切り出す
mosaic = np.zeros(crop.shape[:2], dtype=np.uint8)
mosaic[0::2, 0::2] = crop[0::2, 0::2, 0]           # R
mosaic[0::2, 1::2] = crop[0::2, 1::2, 1]           # G
mosaic[1::2, 0::2] = crop[1::2, 0::2, 1]           # G
mosaic[1::2, 1::2] = crop[1::2, 1::2, 2]           # B
recon = colour_demosaicing.demosaicing_CFA_Bayer_Malvar2004(img_as_float(mosaic), "RGGB")
recon = img_as_ubyte(np.clip(recon, 0, 1))         # 0〜1 に収めて 8 bit に戻す

観測されたベイヤ配列を色つきで表示するため，各画素をその色のチャネルにだけ置く。

In [ ]:
bayerRGB = np.zeros(crop.shape, dtype=np.uint8)
bayerRGB[0::2, 0::2, 0] = mosaic[0::2, 0::2]
bayerRGB[0::2, 1::2, 1] = mosaic[0::2, 1::2]
bayerRGB[1::2, 0::2, 1] = mosaic[1::2, 0::2]
bayerRGB[1::2, 1::2, 2] = mosaic[1::2, 1::2]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], crop, "原画像")
show(axs[1], bayerRGB, "ベイヤ配列の観測")
show(axs[2], recon, "デモザイキング後")
plt.show()

原画像に対するピーク信号対雑音比（PSNR，ピーク値 255）で復元の精度を測る。ライブラリの実装が MATLAB と異なるため，値はスライドとわずかに異なる。

In [ ]:
psnrBayer = peak_signal_noise_ratio(crop, recon, data_range=255)
print(f"PSNR = {psnrBayer:.4f} dB（{psnrBayer:.1f} dB）")

## まとめ
- 明るさの知覚は周囲の影響を受ける（明度対比）。同じ画素値でも同じに見えるとは限らない
- 網膜像は数 mm。視力 1 は 1 分の視角を見分ける能力で，画面高の 3 倍の距離なら走査線数の目安は約 1135 本
- 線形量子化 $y=\lfloor x/Q\rceil$ の階調が少ないと擬似輪郭が現れる
- 動画像のビットレートは $R=3\beta N_1N_2\Delta_\mathrm{t}^{-1}$。8K・60 fps・8 bits なら約 48 Gbps
- カラー画像は R, G, B の 3 成分の組。単板式カメラは 1 画素 1 色しか測らず，残りを補間する

© Copyright, Shogo MURAMATSU, All rights reserved.